# 08 — LLM matcher: bigger model + verify step (Google Colab)

**Goal:** try to improve the LLM matcher of notebook 07 (Qwen 2.5 7B: TEST 23/50 exact, 31/50 CO2e OK, 23 confident errors).

**Two ideas, from the error analysis of notebook 07:**
1. **Bigger model** (`Qwen2.5-14B-Instruct`, 4-bit): most errors were wrong translations, mainly Catalan (*ceba → celery*, *vedella → tomato*).
2. **Verify step:** after choosing, ask "is <chosen food> the same food as <ingredient>?". If no → "uncertain" (no CO2e number shown). Targets absurd picks (*arroz → Carrot, canned*).

**Honesty note:** we already saw the TEST results of notebook 07, so this run is **exploratory**. To keep it fair:
- The 4 configurations (7B / 14B × verify off / on) are compared on the **TUNE split only**.
- **Selection rule, fixed before running:** highest TUNE accuracy; tie → fewer confident errors; tie → smaller model.
- Only the selected configuration is run **once** on TEST. The official result stays 23/50 (notebook 07).

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. It takes about **45–60 minutes** (the 14B model is a ~30 GB download). Keep the tab open.

In [ ]:
# 1. GPU check (should show a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code (branch llm-matcher) and packages
!git clone -q -b llm-matcher https://github.com/laurarigau/Food-Sustainability.git
%cd Food-Sustainability
!pip install -q -r requirements.txt
!pip install -q transformers accelerate bitsandbytes

In [ ]:
# 3. Helpers: evaluate a matcher on TUNE only
import gc, sys, torch, pandas as pd
sys.path.insert(0, "src")
from llm_matching import LLMMatcher, hf_complete
from evaluate_matching import load_labels, predict, apply_threshold, choose_threshold

labels = load_labels()
tune = labels[labels["split"] == "tune"]
GRID = [0.3, 0.6, 1.0]   # score 0 (NONE / rejected by verify) is always uncertain

def tune_scores(matcher):
    pred = predict(matcher, tune)
    threshold, _ = choose_threshold(pred, GRID, "accuracy")
    s = apply_threshold(pred, threshold)
    return {"threshold": threshold, "tune_correct": int(s.correct.sum()), "tune_co2e_ok": int(s.co2e_ok.sum()),
            "tune_confident_errors": int(s.confident_error.sum()), "n": len(s)}

In [ ]:
# 4. Compare the 4 configurations on TUNE (TEST is not used here)
MODELS = ["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-14B-Instruct"]
rows = []
for name in MODELS:
    complete = hf_complete(name)                      # load the model once
    for verify in (False, True):                      # verify=True reuses the cached answers + 1 extra question
        m = LLMMatcher(model=name, provider="hf", complete=complete, threshold=0.0, verify=verify)
        rows.append({"model": name, "verify": verify, **tune_scores(m)})
        print(rows[-1])
    del complete, m
    gc.collect(); torch.cuda.empty_cache()
    slug = name.replace("/", "--")
    !rm -rf ~/.cache/huggingface/hub/models--{slug}    # free disk space for the next model

tune_table = pd.DataFrame(rows)
tune_table

In [ ]:
# 5. Apply the selection rule fixed in advance
tune_table["size"] = tune_table["model"].str.extract(r"(\d+)B").astype(int)
best = tune_table.sort_values(["tune_correct", "tune_confident_errors", "size"],
                              ascending=[False, True, True], kind="stable").iloc[0]
print("Selected on TUNE:", best["model"], "| verify =", best["verify"])

In [ ]:
# 6. Run the selected configuration ONCE on TEST
flag = "--verify" if best["verify"] else ""
cmd = f"python src/evaluate_matching.py --matcher llm --provider hf --model {best['model']} {flag} --details"
print(cmd)
!{cmd}

## Results

Compare with notebook 07 (Qwen 2.5 7B, no verify): **23/50 exact, 31/50 CO2e OK, 23 confident errors**
(AI rev 2: 12 / 19 / 25; RapidFuzz: 7 / 10 / 24). Report this run as **exploratory**.

In [ ]:
# 7. Download the evidence (allow multiple downloads in Chrome if it asks)
from google.colab import files
tune_table.to_csv("data/llm_rev2_tune_search.csv", index=False)
out = f"data/results_llm_{best['model'].split('/')[-1]}{'_verify' if best['verify'] else ''}_test.csv"
print(out)
for f in ["data/llm_rev2_tune_search.csv", out, "data/llm_cache.json"]:
    files.download(f)